# LG-10: LangGraph Studio 可视化调试与 Langfuse 可观测性入门> **阶段**: LG-10 | **难度**: 入门 | **预计时长**: 2-3 小时> **前置要求**: 已完成 LG-01，理解 StateGraph 基础概念## 学习目标- 安装并启动 LangGraph Studio- 在 Studio 中可视化执行图- 掌握断点设置、状态检查、时间旅行调试- 接入 Langfuse，查看 Trace 和瀑布图- 理解 Studio 与 Langfuse 的互补关系---> **开场白**：上一节课你写出了 WeatherBot，代码能跑、结果也对。但有个问题——图跑起来的时候，你其实是"盲"的。> 今天这节课，我们要把图从"黑盒"变成"玻璃盒"。

## 1. 安装依赖LangGraph Studio 需要 langgraph-cli。Langfuse 需要 langfuse Python 包。

In [ ]:
import importlib.metadata# 安装 Studio CLI# !pip install -U "langgraph-cli[inmem]"# 安装 Langfuse SDK# !pip install langfuse# 确保 langgraph 最新版# !pip install -U langgraphimport langgraphprint(f"langgraph 版本: {importlib.metadata.version("langgraph")}")try:    import langfuse    print("langfuse: 已安装")except ImportError:    print("langfuse: 未安装")

## 2. 准备可在 Studio 中运行的图Studio 需要 langgraph.json 配置文件。我们先构建一个带条件分支的 WeatherBot。

In [ ]:
# WeatherBot: 带条件分支的图from typing import TypedDict, Annotatedfrom operator import addfrom datetime import datetimefrom langchain_core.messages import HumanMessagefrom langgraph.graph import StateGraph, START, ENDclass WeatherState(TypedDict):    messages: Annotated[list, add]    intent: str    city: str    weather_data: dict    response: strdef parse_intent(state: WeatherState):    text = state["messages"][-1].content    if "天气" in text or "温度" in text:        return {"intent": "weather"}    elif "时间" in text or "几点" in text:        return {"intent": "time"}    return {"intent": "other"}def extract_city(state: WeatherState):    text = state["messages"][-1].content    city = "北京" if "北京" in text else "上海" if "上海" in text else ""    return {"city": city}def query_weather(state: WeatherState):    return {"weather_data": {"temp": 25, "condition": "晴"}}def format_weather_reply(state: WeatherState):    data = state["weather_data"]    reply = f"{state['city']}今天{data['condition']}，{data['temp']}度"    return {"response": reply}def get_time(state: WeatherState):    return {"response": f"现在是 {datetime.now().strftime('%H:%M')}"}def fallback_reply(state: WeatherState):    return {"response": "抱歉，我暂时只能回答天气和时间相关的问题。"}def route_by_intent(state: WeatherState):    intent = state["intent"]    if intent == "weather":        return "extract_city"    elif intent == "time":        return "get_time"    return "fallback_reply"def route_after_city(state: WeatherState):    if state.get("city"):        return "query_weather"    return "fallback_reply"builder = StateGraph(WeatherState)builder.add_node("parse_intent", parse_intent)builder.add_node("extract_city", extract_city)builder.add_node("query_weather", query_weather)builder.add_node("format_weather_reply", format_weather_reply)builder.add_node("get_time", get_time)builder.add_node("fallback_reply", fallback_reply)builder.add_edge(START, "parse_intent")builder.add_edge("query_weather", "format_weather_reply")builder.add_edge("format_weather_reply", END)builder.add_edge("get_time", END)builder.add_edge("fallback_reply", END)builder.add_conditional_edges("parse_intent", route_by_intent)builder.add_conditional_edges("extract_city", route_after_city)graph = builder.compile()print("Agent 图编译成功！")

### 2.1 图结构可视化在打开 Studio 之前，先用代码生成图的结构图。

In [ ]:
from IPython.display import Image, displaypng_bytes = graph.get_graph().draw_mermaid_png()display(Image(data=png_bytes))print("上图展示了 WeatherBot 的完整图结构")

### 2.2 创建 Studio 项目文件Studio 需要两个文件：1. langgraph.json —— 项目配置文件2. weather_bot.py —— 图定义文件

In [ ]:
import osproject_dir = "/Volumes/DATABASE/code/learn/langgraph_learn/turtorial/LG-10-langgraph-studio-langfuse/examples/10-studio-graph"os.makedirs(project_dir, exist_ok=True)# langgraph.jsonconfig = '{  "dependencies": ["."],  "graphs": {    "weather_bot": "./weather_bot.py:graph"  }}'with open(os.path.join(project_dir, "langgraph.json"), "w") as f:    f.write(config)print(f"项目文件已创建: {project_dir}")print("启动命令: cd {project_dir} && langgraph dev")

## 3. LangGraph Studio 界面导览启动 Studio 后，浏览器自动打开 http://localhost:2024。界面四区：图结构视图、运行配置区、状态面板、时间线/历史。### 操作一：运行图并观察节点高亮1. 输入消息，点击 Run2. 观察节点从 START 开始逐个变绿### 操作二：查看节点 State点击任意已执行节点，状态面板显示 Input State、Output Update、Merged State。

## 4. 核心调试操作### 4.1 时间旅行调试Studio 最强大的能力：不用改代码，直接修改状态重跑。步骤：运行完后点击节点，修改状态，Replay from here。

In [ ]:
print("=== 模拟 Studio 时间旅行调试 ===")print("【第一次运行】正常查询天气")result1 = graph.invoke({"messages": [HumanMessage("北京今天天气怎么样？")]})print(f"  结果: {result1['response']}")print("【时间旅行】修改 intent 后重跑")print("  图会走向不同分支")

### 4.2 打断点在节点上右键 -> Set breakpoint。运行到该节点自动暂停。

## 5. Langfuse 可观测性接入### 5.1 Studio vs LangfuseStudio 是开发时的显微镜，Langfuse 是上线后的仪表盘。| 能力 | Studio | Langfuse ||------|--------|----------|| 实时可视化 | 强 | 弱 || 历史持久化 | 本地 | 数据库 || Token 统计 | 无 | 精确 || 成本分析 | 无 | 有 || 团队协作 | 单机 | 共享 || 生产监控 | 开发工具 | 生产级 |

### 5.2 核心概念- Trace = 一次 graph.invoke()- Span = 一个节点执行- Event = 节点内的具体操作（LLM 调用等）

### 5.3 接入 Langfuse使用用户项目的 Langfuse 配置。

In [ ]:
import osos.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-8969da1a-c681-44fa-a3f1-d1dba9f9cdfb"os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-93b17b96-2324-4dd0-814f-0531b16fb945"os.environ["LANGFUSE_BASE_URL"] = "http://192.168.10.189:3000"pk = os.getenv("LANGFUSE_PUBLIC_KEY")sk = os.getenv("LANGFUSE_SECRET_KEY")if not pk or not sk:    print("环境变量未配置！")else:    print("Langfuse 环境变量已配置")    print(f"  BASE_URL: {os.getenv('LANGFUSE_BASE_URL')}")

### 5.4 创建 CallbackHandler

In [ ]:
import os# 接入 Langfusepk = os.getenv("LANGFUSE_PUBLIC_KEY")sk = os.getenv("LANGFUSE_SECRET_KEY")if pk and sk:    from langfuse.langchain import CallbackHandler    langfuse_handler = CallbackHandler(        session_id="weather-bot-demo",        user_id="student-01",        tags=["env:development", "tutorial:lg-10"]    )    print("CallbackHandler 创建成功")else:    print("跳过（环境变量未配置）")    langfuse_handler = None

### 5.5 使用 Langfuse 运行

In [ ]:
# 使用 Langfuse 运行if langfuse_handler:    print("=== 使用 Langfuse 监控运行 ===")    queries = [        "北京今天天气怎么样？",        "现在几点了？",        "讲个笑话"    ]    for query in queries:        print(f"用户: {query}")        result = graph.invoke(            {"messages": [HumanMessage(query)]},            config={"callbacks": [langfuse_handler]}        )        print(f"Bot: {result['response']}")    print("三次运行已上报到 Langfuse")    print("请前往 http://192.168.10.189:3000 查看")else:    print("跳过运行")    for query in ["北京今天天气怎么样？", "现在几点了？", "讲个笑话"]:        result = graph.invoke({"messages": [HumanMessage(query)]})        print(f"{query} -> {result['response']}")

### 5.6 在 Langfuse 界面看什么？**Trace 列表页**：每次 invoke() 一条记录，耗时、Token 数、成本一眼看到。**Trace 详情页（瀑布图）**：点击任意节点，展开看完整输入输出、Token 用量、模型名称。> 验证：query_weather 显示 1203ms，点进去发现 HTTP Request 占 1198ms。优化方向？> 答：天气 API 响应慢，考虑加缓存或换 API。

### 5.7 Tags 和 Score

In [ ]:
# Tags 和 Scoreif langfuse_handler:    from langfuse import Langfuse    tagged_handler = CallbackHandler(        tags=["env:development", "project:weather-bot"]    )    print("带标签 Handler 创建成功")    print("标签: env:development, project:weather-bot")else:    print("跳过")

## 6. Studio vs Langfuse 选型速查| 场景 | 推荐工具 | 原因 ||------|---------|--------|| 开发阶段调试图逻辑 | Studio | 实时高亮、改状态重跑 || 排查线上问题 | Langfuse | 历史记录、完整链路 || 分析 API 成本 | Langfuse | 精确统计 || 团队协作 | Langfuse | 多用户共享 || 教学演示 | Studio | 界面直观 || 长期监控质量趋势 | Langfuse | Score 时序图 |

## 7. 常见问题**Q：Studio 支持改代码后热重载吗？**A：支持。保存 Python 文件后 Studio 自动重新加载。**Q：Langfuse 数据存在哪里？**A：本项目使用自托管版本 http://192.168.10.189:3000，数据在自己的服务器上。**Q：CallbackHandler 会影响性能吗？**A：overhead 极小，通常在 10ms 以内。

## 8. 课后作业1. 把 WeatherBot 在 Studio 里跑起来，截图三张。2. 去 Langfuse 界面（http://192.168.10.189:3000）找到三次 Trace，回答哪条分支耗时最长。3. **思考题**：Studio 里把 parse_intent 的 intent 从 weather 改成 other，重跑后图会怎么走？---**记忆口诀**：> **Studio 开发看细节，Langfuse 上线看全局。Trace 是家谱，瀑布图找瓶颈，Tags 分环境，Score 量质量。**